In [ ]:
import tensorflow as tf
import numpy as np
import sionna
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray
from sionna.channel import cir_to_ofdm_channel


2026-07-27 07:08:56.483045: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
jitc_llvm_init(): LLVM API initialization failed ..


ModuleNotFoundError: No module named 'sionna.channel'

In [ ]:

# ==========================================
# 1. 씬 및 안테나 배열 셋업
# ==========================================
scene = load_scene(sionna.rt.scene.etoile)
scene.frequency = 3.5e9

# BS: 8x8 UPA (총 64 안테나)
scene.tx_array = PlanarArray(num_rows=8, num_cols=8, 
                             vertical_spacing=0.5, horizontal_spacing=0.5, 
                             pattern="tr38901", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="dipole", polarization="V")

tx = Transmitter(name="BS", position=[0, 0, 25])
scene.add(tx)


In [ ]:

# ==========================================
# 2. 이동 궤적(Trajectory) 설정
# ==========================================
# 예: x=40, y=10~60까지 1m 간격으로 이동하는 50개의 위치 (Batch size = 50)
num_positions = 50
x_pos = tf.fill([num_positions], 40.0)
y_pos = tf.linspace(10.0, 60.0, num_positions)
z_pos = tf.fill([num_positions], 1.5)

# Shape: [num_positions, 3] -> 궤적 텐서 생성
trajectory = tf.stack([x_pos, y_pos, z_pos], axis=1)

# UE 객체 생성 후 속도와 궤적 동시 부여
rx = Receiver(name="UE", position=trajectory) 
rx.velocity = [0.0, 15.0, 0.0]  # y축 방향 15m/s 이동 (도플러 계산용)
scene.add(rx)


In [ ]:

# ==========================================
# 3. 궤적 전체에 대한 Ray-tracing 병렬 연산
# ==========================================
# 50개의 위치에 대한 경로가 동시에 계산됨
paths = scene.compute_paths(max_depth=3)
paths.normalize_delays = False

# ==========================================
# 4. OFDM 주파수 응답 채널 변환
# ==========================================
subcarrier_spacing = 30e3
num_subcarriers = 600
frequencies = tf.cast(tf.linspace(-num_subcarriers/2, num_subcarriers/2 - 1, num_subcarriers) * subcarrier_spacing, tf.float32)

# Shape: [num_positions, num_subcarriers, num_rx_ant, num_tx_ant]
h_freq = cir_to_ofdm_channel(frequencies, paths.a, paths.tau, normalize=True)
h_freq = tf.squeeze(h_freq, axis=2) # Rx 안테나가 1개이므로 스퀴즈 -> [num_positions, num_subcarriers, 64]

# ==========================================
# 5. 공간 공분산 행렬 (Spatial Covariance Matrix) 계산
# ==========================================
# 특정 부반송파(예: 중심 부반송파 index=300)에서의 공간 공분산 분석
h_center = h_freq[:, 300, :] # Shape: [50, 64] (50개 위치의 64개 송신 안테나 채널)

# h_center를 복소수 컬럼 벡터로 취급하여 R = E[h * h^H] 계산
# tf.matmul(h, h, adjoint_a=True) 를 이용해 외적 계산 후 평균화
h_center_expanded = tf.expand_dims(h_center, axis=-1) # [50, 64, 1]
covariance_matrices = tf.matmul(h_center_expanded, h_center_expanded, adjoint_b=True) # [50, 64, 64]

# 전체 궤적(50개 샘플)에 대한 평균 Spatial Covariance Matrix 
R_mean = tf.reduce_mean(covariance_matrices, axis=0) # Shape: [64, 64]